# Data Engineer (итерация 1)

# Data Engineer Report

## Контекст

- Исходный файл: `data/raw/fake_job_postings.csv`
- Цель: подготовить данные для бинарной классификации мошеннических вакансий (target: `fraudulent`)
- Задача: снизить ручную модерацию и защитить пользователей от скам-постингов
- Метрика: F1 / recall класса 1 при контроле precision

## План

1. Загрузить данные и посмотреть общую информацию (размер, пропуски)
2. Провести анализ типов данных, распределения пропусков, распределения таргета
3. Разделить колонки на числовые, категориальные и текстовые
4. Принять решения по очистке:
   - Удалить колонки с >70% пропусков
   - Заполнить пропуски в числовых колонках (mean/median)
   - Заполнить пропуски в категориальных (mode)
   - Обработать выбросы в числовых (клиппинг 1-99 перцентиль)
   - Закодировать категориальные: one-hot для <20 уникальных, frequency encoding для >50
   - Текстовые оставить без изменений
5. Сохранить очищенный датасет в `/Users/nikitayarygin/Documents/projects/ai-agent-gp3/data/processed/cleaned.csv`
6. Отчёт по применённым стратегиям очистки

In [ ]:
import pandas as pd

DF = pd.read_csv("data/raw/fake_job_postings.csv")
print(DF.shape)
print(DF.isna().sum())

(17880, 18)
job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64


## Профиль данных

- Типы данных по колонкам
- Доля пропусков в каждой колонке
- Распределение целевой переменной `fraudulent`
- Разделение колонок на числовые, категориальные и текстовые

In [ ]:
dtypes = DF.dtypes
na_ratio = DF.isna().mean()
target_dist = DF['fraudulent'].value_counts(normalize=True)

num_cols = DF.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = DF.select_dtypes(include=['object']).columns.tolist()

# Определим текстовые колонки по известным именам
text_cols = [col for col in cat_cols if col in ['description', 'requirements', 'benefits', 'company_profile', 'title']]
cat_cols = [col for col in cat_cols if col not in text_cols]

print("Data types:\n", dtypes)
print("NaN ratio:\n", na_ratio)
print("Target distribution:\n", target_dist)
print("Numerical columns:\n", num_cols)
print("Categorical columns:\n", cat_cols)
print("Text columns:\n", text_cols)

<string>:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
Data types:
 job_id                 int64
title                    str
location                 str
department               str
salary_range             str
company_profile          str
description              str
requirements             str
benefits                 str
telecommuting          int64
has_company_logo       int64
has_questions          int64
employment_type          str
required_experience      str
required_education       str
industry                 str
function                 str
fraudulent       

## Стратегия и применение

- Удаляем колонки с более чем 70% пропусков
- Для числовых колонок с выбросами применяем клиппинг по 1 и 99 перцентилям
- Заполняем пропуски в числовых колонках: median для колонок с выбросами, mean для остальных
- Заполняем пропуски в категориальных колонках модой
- Категориальные колонки с <20 уникальных значений кодируем one-hot
- Категориальные колонки с >50 уникальных значений кодируем frequency encoding
- Текстовые колонки оставляем без изменений
- Целевую колонку `fraudulent` не трогаем
- Сохраняем очищенный датасет в указанный путь

In [ ]:
# Удаление колонок с >70% пропусков
cols_to_drop = na_ratio[na_ratio > 0.7].index.tolist()
DF.drop(columns=cols_to_drop, inplace=True)

# Обновим списки колонок после удаления
num_cols = [col for col in num_cols if col not in cols_to_drop]
cat_cols = [col for col in cat_cols if col not in cols_to_drop]
text_cols = [col for col in text_cols if col not in cols_to_drop]

# Обработка числовых колонок
for col in num_cols:
    # Вычислим перцентили
    p1 = DF[col].quantile(0.01)
    p99 = DF[col].quantile(0.99)
    # Клиппинг
    DF[col] = DF[col].clip(lower=p1, upper=p99)
    # Заполнение пропусков
    if DF[col].skew() > 1 or DF[col].skew() < -1:
        # Выбросы есть - медиана
        median_val = DF[col].median()
        DF[col].fillna(median_val, inplace=True)
    else:
        mean_val = DF[col].mean()
        DF[col].fillna(mean_val, inplace=True)

# Обработка категориальных колонок
for col in cat_cols:
    mode_val = DF[col].mode()[0] if not DF[col].mode().empty else ""
    DF[col].fillna(mode_val, inplace=True)

# Кодирование категориальных
# Разделим на две группы по количеству уникальных значений
one_hot_cols = [col for col in cat_cols if DF[col].nunique() < 20]
freq_enc_cols = [col for col in cat_cols if DF[col].nunique() > 50]

# One-hot encoding
DF = pd.get_dummies(DF, columns=one_hot_cols, drop_first=True)

# Frequency encoding
for col in freq_enc_cols:
    freq = DF[col].value_counts(normalize=True)
    DF[col] = DF[col].map(freq)

# Сохраняем очищенный датасет
DF.to_csv("/Users/nikitayarygin/Documents/projects/ai-agent-gp3/data/processed/cleaned.csv", index=False)
print(DF.shape)
print(DF.isna().sum().sum())

<string>:24: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
<string>:21: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting v

## Применённые действия

| Колонка           | Стратегия очистки                 | Причина                                   |
|-------------------|---------------------------------|------------------------------------------|
| Колонки с >70% NaN| Удалены                         | Слишком много пропусков                   |
| Числовые колонки  | Клиппинг 1-99 перцентиль, заполнение mean/median | Устранение выбросов и пропусков           |
| Категориальные <20 уникальных | One-hot encoding          | Избежание потери информации при малом числе уникальных значений |
| Категориальные >50 уникальных | Frequency encoding       | Избежание взрыва размерности при большом числе уникальных значений |
| Текстовые колонки | Оставлены без изменений         | DS самостоятельно решит, как их использовать |
| Целевая колонка `fraudulent` | Не изменялась             | Необходимо сохранить для обучения модели |